# Public review copy — MIS581 Capstone

This notebook is a sanitized publication copy of an executed development notebook. Local Windows paths were replaced with relative placeholders and embedded Census API configuration was changed to use the `CENSUS_API_KEY` environment variable. Historical error outputs were omitted; successful saved outputs were retained. The notebook has **not** been independently re-executed as a clean installation. It may require original source data, a compatible Python environment, and (for Notebook 08) PostgreSQL. Use the project README to understand dependencies and expected folder layout.


In [1]:
import pandas as pd

file_path = r"../data/raw/ssamatab1.xlsx"

df_bls = pd.read_excel(
    file_path,
    header=2
)

In [2]:
print(df_bls.head())
print(df_bls.shape)
print(df_bls.dtypes)
df_bls.info()

         LAUS Code  ST FIPS Code  Area FIPS Code  \
0              NaN           NaN             NaN   
1  MT0111500000000           1.0         11500.0   
2  MT0112220000000           1.0         12220.0   
3  MT0113820000000           1.0         13820.0   
4  MT0119300000000           1.0         19300.0   

                            Area    Year  Month Civilian Labor Force  \
0                            NaN     NaN    NaN                  NaN   
1        Anniston-Oxford, AL MSA  1990.0    1.0                51501   
2         Auburn-Opelika, AL MSA  1990.0    1.0                53578   
3             Birmingham, AL MSA  1990.0    1.0               458343   
4  Daphne-Fairhope-Foley, AL MSA  1990.0    1.0                46132   

  Employment Unemployment Unemployment Rate  
0        NaN          NaN               NaN  
1      47884         3617                 7  
2      49408         4170               7.8  
3     434097        24246               5.3  
4      43694         243

In [3]:
charleston_bls = df_bls[
    df_bls["Area FIPS Code"] == 16700
].copy()

print(charleston_bls.head(20))
print(charleston_bls.tail(20))
print(charleston_bls.shape)

            LAUS Code  ST FIPS Code  Area FIPS Code  \
295   MT4516700000000          45.0         16700.0   
688   MT4516700000000          45.0         16700.0   
1081  MT4516700000000          45.0         16700.0   
1474  MT4516700000000          45.0         16700.0   
1867  MT4516700000000          45.0         16700.0   
2260  MT4516700000000          45.0         16700.0   
2653  MT4516700000000          45.0         16700.0   
3046  MT4516700000000          45.0         16700.0   
3439  MT4516700000000          45.0         16700.0   
3832  MT4516700000000          45.0         16700.0   
4225  MT4516700000000          45.0         16700.0   
4618  MT4516700000000          45.0         16700.0   
5011  MT4516700000000          45.0         16700.0   
5404  MT4516700000000          45.0         16700.0   
5797  MT4516700000000          45.0         16700.0   
6190  MT4516700000000          45.0         16700.0   
6583  MT4516700000000          45.0         16700.0   
6976  MT45

In [4]:
df_bls = pd.read_excel(
    file_path,
    header=2
)

# Keep only actual metropolitan-area observations
df_bls = df_bls[df_bls["Area FIPS Code"].notna()].copy()

print(df_bls.head())
print(df_bls.shape)
print(df_bls.dtypes)
df_bls.info()

         LAUS Code  ST FIPS Code  Area FIPS Code  \
1  MT0111500000000           1.0         11500.0   
2  MT0112220000000           1.0         12220.0   
3  MT0113820000000           1.0         13820.0   
4  MT0119300000000           1.0         19300.0   
5  MT0119460000000           1.0         19460.0   

                            Area    Year  Month Civilian Labor Force  \
1        Anniston-Oxford, AL MSA  1990.0    1.0                51501   
2         Auburn-Opelika, AL MSA  1990.0    1.0                53578   
3             Birmingham, AL MSA  1990.0    1.0               458343   
4  Daphne-Fairhope-Foley, AL MSA  1990.0    1.0                46132   
5                Decatur, AL MSA  1990.0    1.0                65284   

  Employment Unemployment Unemployment Rate  
1      47884         3617                 7  
2      49408         4170               7.8  
3     434097        24246               5.3  
4      43694         2438               5.3  
5      60964         432

In [5]:
bls_measures = [
    "Civilian Labor Force",
    "Employment",
    "Unemployment",
    "Unemployment Rate"
]

for col in bls_measures:
    numeric = pd.to_numeric(df_bls[col], errors="coerce")
    
    nonnumeric = df_bls.loc[
        numeric.isna(),
        ["Area", "Year", "Month", col]
    ]
    
    print(f"\n{col} - Nonnumeric Values")
    print(nonnumeric)
    print("Count:", len(nonnumeric))


Civilian Labor Force - Nonnumeric Values
                                   Area    Year  Month Civilian Labor Force
256                   Cleveland, OH MSA  1990.0    1.0                  (n)
362     Seattle-Tacoma-Bellevue, WA MSA  1990.0    1.0                  (n)
649                   Cleveland, OH MSA  1990.0    2.0                  (n)
755     Seattle-Tacoma-Bellevue, WA MSA  1990.0    2.0                  (n)
1042                  Cleveland, OH MSA  1990.0    3.0                  (n)
...                                 ...     ...    ...                  ...
168980    Racine-Mount Pleasant, WI MSA  2025.0   10.0                  (x)
168981                Sheboygan, WI MSA  2025.0   10.0                  (x)
168982                   Wausau, WI MSA  2025.0   10.0                  (x)
168983                   Casper, WY MSA  2025.0   10.0                  (x)
168984                 Cheyenne, WY MSA  2025.0   10.0                  (x)

[515 rows x 4 columns]
Count: 515

Employment

In [6]:
analysis_nonnumeric = df_bls[
    (df_bls["Year"] >= 2005) &
    (df_bls["Year"] <= 2025) &
    (
        pd.to_numeric(
            df_bls["Unemployment Rate"],
            errors="coerce"
        ).isna()
    )
]

print("\nNonnumeric Unemployment Rate Observations: 2005–2025")
print(
    analysis_nonnumeric
    .groupby(["Year", "Month"])
    .size()
)

print("\nTotal:")
print(len(analysis_nonnumeric))


Nonnumeric Unemployment Rate Observations: 2005–2025
Year    Month
2005.0  9.0        2
        10.0       2
        11.0       2
        12.0       2
2006.0  1.0        2
        2.0        2
        3.0        2
        4.0        2
        5.0        2
        6.0        2
2020.0  3.0        6
        4.0        6
2025.0  10.0     387
dtype: int64

Total:
419


In [7]:
print(
    analysis_nonnumeric[
        analysis_nonnumeric["Year"] < 2025
    ][
        ["Area", "Area FIPS Code", "Year", "Month", "Unemployment Rate"]
    ].to_string(index=False)
)

                                Area  Area FIPS Code   Year  Month Unemployment Rate
        New Orleans-Metairie, LA MSA         35380.0 2005.0    9.0               (n)
Slidell-Mandeville-Covington, LA MSA         43640.0 2005.0    9.0               (n)
        New Orleans-Metairie, LA MSA         35380.0 2005.0   10.0               (n)
Slidell-Mandeville-Covington, LA MSA         43640.0 2005.0   10.0               (n)
        New Orleans-Metairie, LA MSA         35380.0 2005.0   11.0               (n)
Slidell-Mandeville-Covington, LA MSA         43640.0 2005.0   11.0               (n)
        New Orleans-Metairie, LA MSA         35380.0 2005.0   12.0               (n)
Slidell-Mandeville-Covington, LA MSA         43640.0 2005.0   12.0               (n)
        New Orleans-Metairie, LA MSA         35380.0 2006.0    1.0               (n)
Slidell-Mandeville-Covington, LA MSA         43640.0 2006.0    1.0               (n)
        New Orleans-Metairie, LA MSA         35380.0 2006.0    2.

In [8]:
bls_window = df_bls[
    (df_bls["Year"] >= 2005) &
    (df_bls["Year"] <= 2025)
].copy()

print("BLS observations, 2005–2025:", len(bls_window))
print(
    "Unique BLS Area FIPS Codes:",
    bls_window["Area FIPS Code"].nunique()
)

print("\nObservations by year:")
print(
    bls_window.groupby("Year")
    .agg(
        rows=("Area FIPS Code", "size"),
        unique_areas=("Area FIPS Code", "nunique")
    )
)

BLS observations, 2005–2025: 99036
Unique BLS Area FIPS Codes: 393

Observations by year:
        rows  unique_areas
Year                      
2005.0  4716           393
2006.0  4716           393
2007.0  4716           393
2008.0  4716           393
2009.0  4716           393
2010.0  4716           393
2011.0  4716           393
2012.0  4716           393
2013.0  4716           393
2014.0  4716           393
2015.0  4716           393
2016.0  4716           393
2017.0  4716           393
2018.0  4716           393
2019.0  4716           393
2020.0  4716           393
2021.0  4716           393
2022.0  4716           393
2023.0  4716           393
2024.0  4716           393
2025.0  4716           393


In [9]:
df_bls["Unemployment Rate Numeric"] = pd.to_numeric(
    df_bls["Unemployment Rate"],
    errors="coerce"
)

print(df_bls[
    ["Area", "Year", "Month",
     "Unemployment Rate", "Unemployment Rate Numeric"]
].head(10))

print("\nNumeric unemployment rate count:")
print(df_bls["Unemployment Rate Numeric"].count())

print("\nMissing/nonnumeric unemployment rate count:")
print(df_bls["Unemployment Rate Numeric"].isna().sum())

                              Area    Year  Month Unemployment Rate  \
1          Anniston-Oxford, AL MSA  1990.0    1.0                 7   
2           Auburn-Opelika, AL MSA  1990.0    1.0               7.8   
3               Birmingham, AL MSA  1990.0    1.0               5.3   
4    Daphne-Fairhope-Foley, AL MSA  1990.0    1.0               5.3   
5                  Decatur, AL MSA  1990.0    1.0               6.6   
6                   Dothan, AL MSA  1990.0    1.0               4.1   
7   Florence-Muscle Shoals, AL MSA  1990.0    1.0               6.8   
8                  Gadsden, AL MSA  1990.0    1.0               9.9   
9               Huntsville, AL MSA  1990.0    1.0               5.2   
10                  Mobile, AL MSA  1990.0    1.0               6.9   

    Unemployment Rate Numeric  
1                         7.0  
2                         7.8  
3                         5.3  
4                         5.3  
5                         6.6  
6                         

In [10]:
bls_analysis = df_bls[
    (df_bls["Year"] >= 2005) &
    (df_bls["Year"] <= 2025)
].copy()

print("\nBLS Analysis Period: 2005–2025")
print(bls_analysis["Unemployment Rate Numeric"].describe())

print("\nAdditional Unemployment Rate Statistics")
print("Median:", bls_analysis["Unemployment Rate Numeric"].median())
print("Variance:", bls_analysis["Unemployment Rate Numeric"].var())
print("Skewness:", bls_analysis["Unemployment Rate Numeric"].skew())

print("\nAnalysis Dataset Shape")
print(bls_analysis.shape)


BLS Analysis Period: 2005–2025
count    98617.000000
mean         5.832666
std          2.970396
min          1.600000
25%          3.800000
50%          5.000000
75%          7.100000
max         34.300000
Name: Unemployment Rate Numeric, dtype: float64

Additional Unemployment Rate Statistics
Median: 5.0
Variance: 8.823250288959933
Skewness: 2.060244329786622

Analysis Dataset Shape
(99036, 11)
